
# **Merve Arta - 20/03/2025**


**Bu proje mağaza satışlarının zaman serisi verileri üzerinden tahmin yapmaya yönelik bir yaklaşım içeriyor.**

**Amaç: Mağaza satışlarının zaman serisi verisini kullanarak gelecekteki satışları tahmin etmek.**

In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import ListedColormap
from statsmodels.tsa.deterministic import CalendarFourier, DeterministicProcess
from sklearn.linear_model import LinearRegression
from pandas import date_range
from statsmodels.graphics.tsaplots import plot_pacf
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor
from sklearn.preprocessing import LabelEncoder
#from pyearth import Earth
from sklearn.linear_model import ElasticNet, Lasso, Ridge
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.multioutput import RegressorChain


* train.csv: Ana eğitim verisi (tarih, mağaza numarası, ürün ailesi, satış, vb.)
* test.csv: Yarışma için tahminlenecek veriler (tarih, mağaza numarası, ürün ailesi, vb.)
* stores.csv: Mağazalara ait meta veriler (mağaza tipi, bölge, vb.)
* transactions.csv: Mağaza bazlı günlük işlem sayıları
* holidays_events.csv: Tatiller ve özel günler bilgisi
* oil.csv: Petrol fiyatları (ülke ekonomisini etkileyen bir faktör olarak kullanılabilir)
* sample_submission.csv: Örnek tahmin formatı




**“Neden sadece train ile oil dosyalarını kullanacağız, diğer dosyaları neden dahil etmiyoruz?” Aslında amacımız tüm veri setlerini (stores, transactions, holidays vs.) birleştirip (merge) zengin bir veri tablosu oluşturmak; ancak her şeyi tek seferde göstermek karmaşık olabiliyor.**


**Bizim, asıl veri setimiz train.csv olduğu için sonraki veri ön işleme işlemlerini ona göre yapacağız.**

# Diğer CSV Dosyalarıyla Birleştirme (Merge) ve Özellikleri Eklemek

In [ ]:
holidays_df  = pd.read_csv("/kaggle/input/store-sales-time-series-forecasting/holidays_events.csv")
oil_df  = pd.read_csv("/kaggle/input/store-sales-time-series-forecasting/oil.csv")
stores_df  = pd.read_csv("/kaggle/input/store-sales-time-series-forecasting/stores.csv")
transactions_df  = pd.read_csv("/kaggle/input/store-sales-time-series-forecasting/transactions.csv")
train_df  = pd.read_csv("/kaggle/input/store-sales-time-series-forecasting/train.csv")
test_df  = pd.read_csv("/kaggle/input/store-sales-time-series-forecasting/test.csv")

# İlk bakış

In [ ]:

print("train_df ilk 5 satır:\n", train_df.head())
print("\nstores_df ilk 5 satır:\n", stores_df.head())
print("\ntransactions_df ilk 5 satır:\n", transactions_df.head())
print("\nholidays_df ilk 5 satır:\n", holidays_df.head())
print("\noil_df ilk 5 satır:\n", oil_df.head())

# Veri Hakkında Bilgi Edinelim

In [ ]:

print("\ntrain_df boyut:", train_df.shape)
print("\ntrain_df sütun bilgileri:")
print(train_df.info())

print("\ntrain_df özet istatistikler:")
print(train_df.describe())


# Dosyaları Okuma ve Tarih Dönüşümleri
**Önce her dosyayı Pandas ile okuyup tarih sütununu datetime formatına çevirelim.**

In [ ]:
train_df['date'] = pd.to_datetime(train_df['date'])
test_df['date'] = pd.to_datetime(test_df['date'])
oil_df['date'] = pd.to_datetime(oil_df['date'])
holidays_df['date'] = pd.to_datetime(holidays_df['date'])
transactions_df['date'] = pd.to_datetime(transactions_df['date'])


# Daha zengin bir veri seti elde etmek için tüm datalarımızı birleştirelim.


**stores.csv dosyasını train.df içine ekleyelim.Bu sayede train.df içine mağazalara ait farklı bilgileri de bulundurabiliriz.İki veri seti içindeki ortak sütun store_nbr kolunu**


In [ ]:
# Kolonları görelim
print(train_df.columns)
print(stores_df.columns)

In [ ]:
# Birleştirme
train_merged = pd.merge(
    train_df, 
    stores_df, 
    on='store_nbr',   # ortak sütun
    how='left'        # train_df'deki tüm satırlar kalsın
)

In [ ]:
print("Birleşim sonrası sütunlar:", train_merged.columns)
print(train_merged.head())

**Mağaza bazında günlük işlem sayılarını (transactions) da eklemek.** 
**Bu da satışları etkileyen önemli bir değişken olabilir.**

In [ ]:
# train_merged ve transactions_df sütunlarına bakalım
print(train_merged.columns)
print(transactions_df.columns)

**store_nbr kolonu yine bu veri seti ile ortak bir kolon yine bir birleştirme işlemi yapabiliriz.**

In [ ]:

# Birleştirme
train_merged = pd.merge(
    train_merged, 
    transactions_df, 
    on=['date', 'store_nbr'], 
    how='left'
)

In [ ]:
print("Birleşim sonrası sütunlar:", train_merged.columns)
print(train_merged.head())

In [ ]:
# train_merged ve holiday_df sütunlarına bakalım
print(train_merged.columns)
print(holidays_df.columns)

In [ ]:
holidays_df.head()

**Sadece ulusal tatiller için (locale = National) her mağazaya aynı etkiyi vermek mantıklı olabilir.**
**Depo bazında yapılan aktiviteler ya da bayram tatilleri her bölgeyi etkileyemez.**

In [ ]:
# Sadece "National" tatilleri birleştirelim
nat_holidays = holidays_df[holidays_df['locale'] == 'National']

In [ ]:
# Sütun ismi çakışması olmaması için rename ile "type" -> "holiday_type" yapabilirsiniz
nat_holidays = nat_holidays.rename(columns={'type': 'holiday_type'})

In [ ]:
nat_holidays.head()

In [ ]:
print(train_merged.columns)

In [ ]:
train_merged = pd.merge(
    train_merged,
    nat_holidays[['date', 'holiday_type']],  # sadece date ve holiday_type alalım
    on='date',
    how='left'
)


In [ ]:
# Artık "holiday_type" sütunu varsa, NaN olmayan satırlar ulusal tatil günleri
print(train_merged[['date','holiday_type']].drop_duplicates().head(10))

In [ ]:
# train_merged ve holidays_df sütunlarına bakalım
print(train_merged.columns)
print(holidays_df.columns)

In [ ]:
print(train_merged.columns)
print(oil_df.columns)

In [ ]:
print("\noil_df ilk 5 satır:\n", oil_df.head())

In [ ]:
# Eksik değerleri dolduralım (forward fill vb.)
oil_df['dcoilwtico'].fillna(method='ffill', inplace=True)

In [ ]:
train_merged = pd.merge(
    train_merged,
    oil_df[['date','dcoilwtico']],  # sadece gerekli sütunlar
    on='date',
    how='left'
)

In [ ]:
train_merged

In [ ]:
# Veri setinizin boyutunu kontrol edin:
train_merged.shape

In [ ]:
# Temel istatistiksel özet:
print("\nTemel İstatistiksel Özet:")
train_merged.describe()

In [ ]:
# Veri tipi ve sütun bilgileri:
print("\nSütun Bilgileri:")
train_merged.info()


# Eksik Değerleri ve Dağılımı İnceleyelim

In [ ]:
# Her sütundaki eksik değer sayısını kontrol edin:
print("\nEksik Değerler:")
train_merged.isnull().sum()


**EDA aşamasında eksik değerleri inceleyip nedenlerini ve dağılımlarını anlamak, hangi imputation yönteminin en uygun olduğuna karar vermek için kritik bir adımdır.**
**Temizleme ve doldurma işlemlerini, bu analiz sonuçlarına göre daha sonraki aşamalarda gerçekleştireceğiz.**

# Zaman Serisi Grafiği – Genel Trend

In [ ]:
# Tarihe göre toplam satışları hesaplayalım:
daily_sales = train_merged.groupby('date')['sales'].sum()

plt.figure(figsize=(12,6))
plt.plot(daily_sales.index, daily_sales.values, label="Günlük Toplam Satış")
plt.title("Zaman İçinde Günlük Toplam Satışlar")
plt.xlabel("Tarih")
plt.ylabel("Satış")
plt.legend()
plt.show()

# Segment Bazında Analiz – Mağaza ve Ürün Aileleri

In [ ]:
train_merged['store_nbr']

In [ ]:
train_merged['family']

In [ ]:
# Örnek: Mağaza 1 ve "GROCERY I" ürün ailesi için
store_nbr = 1
product_family = "BEVERAGES"

subset_df = train_merged[(train_merged['store_nbr'] == store_nbr) & 
                         (train_merged['family'] == product_family)]
daily_subset_sales = subset_df.groupby('date')['sales'].sum()

plt.figure(figsize=(12,6))
plt.plot(daily_subset_sales.index, daily_subset_sales.values, label=f"Mağaza {store_nbr} - {product_family}")
plt.title(f"Mağaza {store_nbr} ve {product_family} Ürün Ailesi Günlük Satışlar")
plt.xlabel("Tarih")
plt.ylabel("Satış")
plt.legend()
plt.show()

# Tarih Tabanlı Özellikler (Feature Engineering)

In [ ]:
# Tarih sütunundan ek özellikler oluşturma:
train_merged['year'] = train_merged['date'].dt.year
train_merged['month'] = train_merged['date'].dt.month
train_merged['day'] = train_merged['date'].dt.day
train_merged['day_of_week'] = train_merged['date'].dt.dayofweek  # Pazartesi=0, Pazar=6
train_merged['is_weekend'] = train_merged['day_of_week'].apply(lambda x: 1 if x >= 5 else 0)


In [ ]:
train_merged[['date', 'year', 'month', 'day', 'day_of_week', 'is_weekend']]

# Tatil ve Özel Günler

In [ ]:
# Eğer daha önce holidays_df'den ulusal tatil bilgisi eklediyseniz:
# 'holiday_type' sütununa göre bayrak sütunu oluşturma:
train_merged['is_holiday'] = train_merged['holiday_type'].notnull().astype(int)

# Tatil bilgilerini içeren birkaç satırı görüntüleyin:
print(train_merged[['date', 'holiday_type', 'is_holiday']].drop_duplicates().head(10))

# Korelasyon Analizi

In [ ]:
# İncelenecek sütunlar: satış, işlem sayısı, petrol fiyatı, tatil bayrağı, hafta sonu, ay vb.
corr = train_merged[['sales', 'transactions', 'dcoilwtico', 'is_holiday', 'is_weekend', 'month']].corr()

plt.figure(figsize=(8,6))
sns.heatmap(corr, annot=True, cmap="coolwarm")
plt.title("Özellikler Arası Korelasyon")
plt.show()

# sales - transactions (0.21):

**0.21’lik bir korelasyon, pozitif yönde ama düşük/orta seviyede bir ilişki olduğunu gösterir.**
**Yani işlem sayısı arttığında satışlar da artma eğiliminde, ancak çok güçlü bir ilişki değil.**
**Buna rağmen, tablo içindeki en yüksek korelasyon bu olduğu için, “transactions” değişkeninin “sales” tahmininde diğerlerinden daha faydalı olabileceğini düşünebilirsiniz.**

# Eksik Verileri Doldurma (Missing Data Imputation)

In [ ]:
# Her sütundaki eksik değer sayısını kontrol edin:
print("\nEksik Değerler:")
train_merged.isnull().sum()

In [ ]:
train_merged['transactions']

In [ ]:
train_merged['holiday_type']

In [ ]:
train_merged['dcoilwtico']

In [ ]:
# NaN olanları "No Holiday" olarak işaretleyelim
train_merged['holiday_type'].fillna("No Holiday", inplace=True)

In [ ]:
# Mağaza bazında ortalama hesaplayıp doldurma
train_merged['transactions'] = (
    train_merged.groupby('store_nbr')['transactions']
                .transform(lambda x: x.fillna(x.mean()))
)



In [ ]:
train_merged['dcoilwtico'] = train_merged['dcoilwtico'].ffill().bfill()

# ffill() (Forward Fill):


**Bu metod, sütundaki her NaN değeri, kendisinden önceki geçerli (non-NaN) değerin kopyasıyla doldurur.Örneğin, eğer 5. satırda NaN varsa ve 4. satırda bir değer varsa, 5. satır o değeri alır.bfill() (Backward Fill):Forward fill işleminden sonra hala doldurulamayan (örneğin, sütunun en başındaki NaN'ler) değerler için, bfill() metodu kullanılır.Bu metod, NaN değerleri, kendisinden sonraki geçerli değerin kopyasıyla doldurur.**


In [ ]:
print("\nEksik Değerler (Doldurma Sonrası):")
train_merged.isnull().sum()

# Modellemeye Geçelim

# Linear Regression

**Basit bir doğrusal regresyon modeli kurarak, temel özelliklerin (transactions, dcoilwtico, is_holiday, is_weekend, month, day_of_week vb.) satışları (sales) nasıl etkilediğini görebiliriz.**

In [ ]:
# Örnek: Zaman bazlı train-test split
split_date = train_merged['date'].quantile(0.8)
train_data = train_merged[train_merged['date'] <= split_date]
test_data = train_merged[train_merged['date'] > split_date]

print("Eğitim verisi boyutu:", train_data.shape)
print("Test verisi boyutu:", test_data.shape)


Quantile : yüzde (80th percentile) tarih değerini hesaplar. Yani, veri setinizdeki tarihleri sıraladığınızda, %80'inin bu tarihten önce, %20'sinin ise bu tarihten sonra olduğunu belirleyen eşik tarihini verir.

In [ ]:
features = ['transactions', 'dcoilwtico', 'is_holiday', 'is_weekend', 'month', 'day_of_week']
target = 'sales'

X_train = train_data[features]
y_train = train_data[target]
X_test = test_data[features]
y_test = test_data[target]

print("Özellikler:", X_train.head())
print("Hedef:", y_train.head())


In [ ]:
from sklearn.linear_model import LinearRegression

# Model oluşturma
lr_model = LinearRegression()

# Modeli eğitim verisi üzerinde eğitme
lr_model.fit(X_train, y_train)

# Eğitim seti üzerinde modelin katsayılarına bakalım
print("Model Katsayıları:", lr_model.coef_)
print("Model Sabiti (Intercept):", lr_model.intercept_)


In [ ]:
from sklearn.metrics import mean_squared_error
import numpy as np

# Test verisi için tahmin
y_pred = lr_model.predict(X_test)

# RMSE hesaplama
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print("Linear Regression Test RMSE:", rmse)


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12,6))
plt.plot(test_data['date'], y_test, label="Gerçek Satışlar", color='blue')
plt.plot(test_data['date'], y_pred, label="Tahmin Edilen Satışlar", color='red', alpha=0.7)
plt.xlabel("Tarih")
plt.ylabel("Satış")
plt.title("Gerçek ve Tahmin Edilen Satışlar")
plt.legend()
plt.show()


# XGBoost

**Doğrusal olmayan ilişkileri yakalamada ve karmaşık veri dinamiklerini modellemede genellikle daha başarılıdır.**
**Basit doğrusal regresyon modelinden farklı olarak, XGBoost karar ağaçları üzerinden gradyan yükseltme yöntemini kullanır.** 

In [ ]:
import xgboost as xgb
import numpy as np
from sklearn.metrics import mean_squared_error

# Özellikler ve hedef değişkenler (daha önce tanımlandı)
# X_train, y_train, X_test, y_test

# XGBoost DMatrix formatına dönüştürme:
dtrain = xgb.DMatrix(X_train, label=y_train)
dtest = xgb.DMatrix(X_test, label=y_test)


In [ ]:
params = {
    'objective': 'reg:squarederror',  # Regresyon problemi için kare hata minimizasyonu
    'eval_metric': 'rmse',
    'eta': 0.1,
    'max_depth': 5,
    'seed': 42
}

In [ ]:
watchlist = [(dtrain, 'train'), (dtest, 'eval')]
num_rounds = 100

xgb_model = xgb.train(params, 
                      dtrain, 
                      num_boost_round=num_rounds, 
                      evals=watchlist, 
                      early_stopping_rounds=10, 
                      verbose_eval=True)

In [ ]:
y_pred_xgb = xgb_model.predict(dtest)
rmse_xgb = np.sqrt(mean_squared_error(y_test, y_pred_xgb))
print("XGBoost Test RMSE:", rmse_xgb)


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12,6))
plt.plot(test_data['date'], y_test, label="Gerçek Satışlar", color='blue')
plt.plot(test_data['date'], y_pred_xgb, label="XGBoost Tahminleri", color='red', alpha=0.7)
plt.xlabel("Tarih")
plt.ylabel("Satış")
plt.title("Gerçek Satışlar vs XGBoost Tahminleri")
plt.legend()
plt.show()


In [ ]:
# RandomForestRegressor
from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_pred))
print("RandomForestRegressor Test RMSE:", rf_rmse)


In [ ]:
# LightGBM
import lightgbm as lgb

# LightGBM verilerini oluşturuyoruz
lgb_train = lgb.Dataset(X_train, label=y_train)

lgb_params = {
    'objective': 'regression',
    'metric': 'rmse',
    'boosting_type': 'gbdt',
    'learning_rate': 0.1,
    'num_leaves': 31,
    'seed': 42
}

lgb_model = lgb.train(lgb_params, lgb_train, num_boost_round=100)
lgb_pred = lgb_model.predict(X_test)
lgb_rmse = np.sqrt(mean_squared_error(y_test, lgb_pred))
print("LightGBM Test RMSE:", lgb_rmse)

In [ ]:
# CatBoostRegressor
from catboost import CatBoostRegressor

cat_model = CatBoostRegressor(iterations=100, learning_rate=0.1, depth=6, random_seed=42, verbose=0)
cat_model.fit(X_train, y_train)
cat_pred = cat_model.predict(X_test)
cat_rmse = np.sqrt(mean_squared_error(y_test, cat_pred))
print("CatBoostRegressor Test RMSE:", cat_rmse)


In [ ]:
# (Opsiyonel) Gerçek ve tahmin değerlerini karşılaştıran basit bir grafik çizelim:
plt.figure(figsize=(12,6))
plt.plot(test_data['date'], y_test, label="Gerçek Satışlar", color='blue')
plt.plot(test_data['date'], rf_pred, label="RF Tahminleri", alpha=0.7)
plt.plot(test_data['date'], lgb_pred, label="LightGBM Tahminleri", alpha=0.7)
plt.plot(test_data['date'], cat_pred, label="CatBoost Tahminleri", alpha=0.7)
plt.xlabel("Tarih")
plt.ylabel("Satış")
plt.title("Gerçek Satışlar ve Farklı Modellerin Tahminleri")
plt.legend()
plt.show()